# Part 2A-viii: Custom Layers

**Objective:** Build custom layers including Exponential, Dense, GaussianNoise, and LayerNormalization.

---

In [1]:
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch, torch.nn as nn

(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
X_train, X_test = X_train.astype("float32")/255.0, X_test.astype("float32")/255.0
y_train, y_test = y_train.flatten(), y_test.flatten()
X_flat_tr, X_flat_te = X_train.reshape(len(X_train),-1), X_test.reshape(len(X_test),-1)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


In [2]:
# TF: Custom Exponential Layer (no weights — pure computation)
class ExponentialLayer(layers.Layer):
    """Applies element-wise exponential transformation. Useful as output layer for positive-valued predictions."""
    def call(self, inputs):
        return tf.exp(inputs)

# TF: Custom Dense Layer (from scratch)
class MyDense(layers.Layer):
    """Dense layer built from scratch — demonstrates manual weight management."""
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.kernel = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer="glorot_normal", trainable=True, name="kernel"
        )
        self.bias = self.add_weight(
            shape=(self.units,),
            initializer="zeros", trainable=True, name="bias"
        )

    def call(self, inputs):
        z = tf.matmul(inputs, self.kernel) + self.bias
        return self.activation(z) if self.activation else z

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units, "activation": keras.activations.serialize(self.activation)})
        return config

In [3]:
# TF: Gaussian Noise Layer (training-only noise injection)
class AddGaussianNoise(layers.Layer):
    """Adds Gaussian noise during training — acts as regularization.
    At inference, no noise is added.
    """
    def __init__(self, stddev=0.1, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self, inputs, training=False):
        if training:
            noise = tf.random.normal(tf.shape(inputs), mean=0.0, stddev=self.stddev)
            return inputs + noise
        return inputs

# TF: Custom Layer Normalization
class MyLayerNormalization(layers.Layer):
    """Layer Normalization: normalizes across features (not batch).
    Unlike BatchNorm, works the same during training and inference.
    """
    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        self.gamma = self.add_weight(
            shape=(input_shape[-1],), initializer="ones", name="gamma"
        )
        self.beta = self.add_weight(
            shape=(input_shape[-1],), initializer="zeros", name="beta"
        )

    def call(self, inputs):
        mean = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        var = tf.reduce_mean(tf.square(inputs - mean), axis=-1, keepdims=True)
        normalized = (inputs - mean) / tf.sqrt(var + self.epsilon)
        return self.gamma * normalized + self.beta

In [4]:
# Build model using ALL custom layers
model = keras.Sequential([
    layers.Input(shape=(3072,)),
    MyDense(256, activation='relu'),
    AddGaussianNoise(0.1),
    MyLayerNormalization(),
    MyDense(128, activation='relu'),
    AddGaussianNoise(0.05),
    MyLayerNormalization(),
    MyDense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
h = model.fit(X_flat_tr, y_train, epochs=20, batch_size=256, validation_split=0.2, verbose=1)
loss, acc = model.evaluate(X_flat_te, y_test, verbose=0)
print(f"\nTest Accuracy with custom layers: {acc:.4f}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ my_dense (MyDense)              │ (None, 256)            │       786,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_gaussian_noise              │ (None, 256)            │             0 │
│ (AddGaussianNoise)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_layer_normalization          │ (None, 256)            │           512 │
│ (MyLayerNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_dense_1 (MyDense)            │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_gaussian_noise_1            │ (None, 128)            │             0 │
│ (AddGaussianNoise)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_layer_normalization_1        │ (None, 128)            │           256 │
│ (MyLayerNormalization)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_dense_2 (MyDense)            │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 821,642 (3.13 MB)

 Trainable params: 821,642 (3.13 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.2315 - loss: 2.1219 - val_accuracy: 0.3110 - val_loss: 1.9216
Epoch 2/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3446 - loss: 1.8410 - val_accuracy: 0.3618 - val_loss: 1.7830
Epoch 3/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3911 - loss: 1.7229 - val_accuracy: 0.3808 - val_loss: 1.7539
Epoch 4/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4162 - loss: 1.6424 - val_accuracy: 0.3903 - val_loss: 1.7086
Epoch 5/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4380 - loss: 1.5902 - val_accuracy: 0.4366 - val_loss: 1.5913
Epoch 6/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4494 - loss: 1.5490 - val_accuracy: 0.4189 - val_loss: 1.6583
Epoch 7/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4602 - loss: 1.5131 - val_accuracy: 0.4485 - val_loss: 1.5559
Epoch 8/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4736 - loss: 1.4817 - val_accuracy: 0

In [5]:
# PyTorch: Custom layers
class MyDensePT(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_f, out_f) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_f))
    def forward(self, x): return x @ self.weight + self.bias

class GaussianNoisePT(nn.Module):
    def __init__(self, std=0.1): super().__init__(); self.std = std
    def forward(self, x):
        if self.training: return x + torch.randn_like(x) * self.std
        return x

class LayerNormPT(nn.Module):
    def __init__(self, features, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / (var + self.eps).sqrt() + self.beta

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pt_model = nn.Sequential(
    MyDensePT(3072, 256), nn.ReLU(), GaussianNoisePT(0.1), LayerNormPT(256),
    MyDensePT(256, 128), nn.ReLU(), GaussianNoisePT(0.05), LayerNormPT(128),
    MyDensePT(128, 10)
).to(device)

opt = torch.optim.Adam(pt_model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.FloatTensor(X_flat_tr), torch.LongTensor(y_train)),
    batch_size=256, shuffle=True)
for ep in range(20):
    pt_model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); crit(pt_model(xb), yb).backward(); opt.step()
pt_model.eval()
with torch.no_grad():
    acc = (pt_model(torch.FloatTensor(X_flat_te).to(device)).argmax(1)==torch.LongTensor(y_test).to(device)).float().mean()
print(f"PyTorch Custom Layers Test Acc: {acc:.4f}")

PyTorch Custom Layers Test Acc: 0.5034


## Key Takeaways
- **ExponentialLayer**: stateless computation layer (no weights)
- **MyDense**: manual weight creation with `add_weight()` / `nn.Parameter`
- **AddGaussianNoise**: training-only regularization via noise injection
- **MyLayerNormalization**: per-sample normalization (unlike BatchNorm which normalizes across batch)
- Custom layers compose naturally with built-in layers in Sequential or Functional API